# Chapter 11 — Training LLM Agents with RL## GAE Implementation and Code Agent Training[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**GPU recommended. Runtime: ~30 minutes.**Implements:1. Full GAE computation for multi-step agent episodes2. A minimal REINFORCE code agent with verifiable rewards3. The GRPO group-relative advantage estimation

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
print('Ready')

## 11.1 GAE for Agent EpisodesIn Chapter 11 we showed GAE applied to a 50-step agent episode. Here we implement it and verify it distributes credit correctly.

In [ ]:
def compute_gae(rewards, values, next_values, dones, gamma=0.99, lam=0.95):
    """
    Compute GAE advantages for an agent episode.
    Args:
        rewards:     list of rewards r_t
        values:      V(s_t) critic estimates
        next_values: V(s_{t+1}) critic estimates
        dones:       episode termination flags
    Returns:
        normalised advantages tensor
    """
    T = len(rewards)
    advantages = [0.0] * T
    gae = 0.0
    for t in reversed(range(T)):
        next_val = 0.0 if dones[t] else next_values[t]
        delta = rewards[t] + gamma * next_val - values[t]
        gae = delta + gamma * lam * (0.0 if dones[t] else gae)
        advantages[t] = gae
    adv_tensor = torch.FloatTensor(advantages)
    return (adv_tensor - adv_tensor.mean()) / (adv_tensor.std() + 1e-8)

# Simulate a 30-step episode with a critical event at step 10
T = 30
rewards = [0.0] * T
rewards[29] = 1.0  # sparse terminal reward
values = [0.05] * T + [0.0]  # critic slightly above zero
next_values = values[1:]
dones = [False] * 29 + [True]

advantages = compute_gae(rewards, values[:-1], next_values, dones, gamma=0.99, lam=0.95)

fig, ax = plt.subplots(figsize=(10,4))
ax.bar(range(T), advantages.numpy(), color=['steelblue' if a>0 else 'tomato' for a in advantages])
ax.set_xlabel('Step t')
ax.set_ylabel('GAE Advantage')
ax.set_title('GAE Credit Distribution: Sparse Reward at Step 30')
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 11.2 Code Agent EnvironmentA simplified code agent: receives a Python problem, generates a solution, receives reward based on test pass rate.

In [ ]:
import subprocess, tempfile, os

SAMPLE_PROBLEMS = [
    {
        'description': 'Write a function add(a, b) that returns the sum of a and b.',
        'test_cases': [
            {'call': 'add(2, 3)',   'expected': '5'},
            {'call': 'add(-1, 1)', 'expected': '0'},
            {'call': 'add(0, 0)',  'expected': '0'},
        ]
    },
    {
        'description': 'Write a function is_palindrome(s) that returns True if s is a palindrome.',
        'test_cases': [
            {'call': 'is_palindrome("racecar")', 'expected': 'True'},
            {'call': 'is_palindrome("hello")',   'expected': 'False'},
            {'call': 'is_palindrome("a")',       'expected': 'True'},
        ]
    },
]

def run_test(code, test):
    full_code = f'{code}\nresult = {test["call"]}\nprint(result)'
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(full_code); fname = f.name
    try:
        proc = subprocess.run(['python3', fname], capture_output=True, text=True, timeout=5)
        passed = proc.stdout.strip() == test['expected'] and proc.returncode == 0
        return passed
    except:
        return False
    finally:
        os.unlink(fname)

def compute_reward(code, problem):
    n_passed = sum(run_test(code, t) for t in problem['test_cases'])
    return n_passed / len(problem['test_cases'])

# Test the reward
code_correct = 'def add(a, b): return a + b'
code_wrong   = 'def add(a, b): return a - b'
print(f'Correct code reward: {compute_reward(code_correct, SAMPLE_PROBLEMS[0]):.2f}')
print(f'Wrong code reward:   {compute_reward(code_wrong,   SAMPLE_PROBLEMS[0]):.2f}')

## 11.3 GRPO: Group-Relative Advantage EstimationGRPO eliminates the value function by using the group of responses as its own baseline.

In [ ]:
def grpo_advantages(rewards_group):
    """
    Compute GRPO advantages from a group of G responses to the same prompt.
    rewards_group: list of G scalar rewards
    returns: normalised advantages
    """
    rewards = torch.FloatTensor(rewards_group)
    mean_r = rewards.mean()
    std_r  = rewards.std() + 1e-8
    return (rewards - mean_r) / std_r

# Example: 8 responses to the same coding problem
group_rewards = [0.0, 0.33, 0.33, 0.67, 0.67, 0.67, 1.0, 1.0]
advantages = grpo_advantages(group_rewards)

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].bar(range(len(group_rewards)), group_rewards, color='steelblue')
axes[0].set_title('Group Rewards (G=8 responses)')
axes[0].set_xlabel('Response index')
axes[0].set_ylabel('Reward')
axes[1].bar(range(len(advantages)), advantages.numpy(),
            color=['seagreen' if a>0 else 'tomato' for a in advantages])
axes[1].set_title('GRPO Advantages (normalised)')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Response index')
plt.tight_layout()
plt.show()
print('Group rewards:', group_rewards)
print('GRPO advantages:', advantages.numpy().round(3))

## ✅ Key Takeaways1. GAE distributes sparse terminal rewards backward through the episode2. λ=0.95 balances variance (low λ → more bias but less noise) and bias3. GRPO eliminates the value function: advantage = (r - mean) / std within the group4. Verifiable rewards (test pass rate) provide clean, ungameable training signal